# Tarea C4 — Transfer Learning en Dos Fases
**ECG Rhythm Analyzer | Gabriela — Modelo CNN**  
**Rama:** `feature/model-training`

## Objetivo
Implementar el entrenamiento en **dos fases**:
- **Fase 1:** Solo entrenar la última capa (las demás congeladas) — 5 epochs
- **Fase 2:** Descongelar las últimas capas y afinar todo junto con lr muy bajo — 5 epochs

También se añade **ReduceLROnPlateau**: reduce el learning rate automáticamente cuando el modelo se estanca.

## ¿Por qué dos fases?
EfficientNet-B0 ya sabe detectar bordes, texturas y patrones gracias a ImageNet. Si desde el inicio entrenamos todas las capas con lr alto, podemos "romper" ese conocimiento previo. La estrategia de dos fases protege ese conocimiento:
- **Fase 1:** La nueva capa aprende qué buscar en las imágenes de ECG
- **Fase 2:** Todo el modelo se afina suavemente para adaptarse mejor a ECGs


## PASO 0 — Configuración

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import os
from PIL import Image
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE   = 224
BATCH_SIZE = 32
BASE_DIR   = 'ecg_dataset_dummy'
CLASES     = ['normal', 'arritmia']
SPLITS     = ['train', 'val', 'test']

print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## PASO 1 — Recrear datos dummy y data loaders
*(Igual que C3 — necesario porque cada sesión de Colab empieza limpia)*

In [ ]:
# Recrear datos dummy
SPLIT_SIZES = {'train': 70, 'val': 20, 'test': 10}
np.random.seed(42)

for split in SPLITS:
    for clase in CLASES:
        os.makedirs(os.path.join(BASE_DIR, split, clase), exist_ok=True)

for split, n_imgs in SPLIT_SIZES.items():
    for clase in CLASES:
        folder = os.path.join(BASE_DIR, split, clase)
        if len(os.listdir(folder)) == 0:
            for i in range(n_imgs):
                pixels = np.random.randint(0, 255, (IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
                Image.fromarray(pixels, 'RGB').save(os.path.join(folder, f'ecg_{split}_{clase}_{i:04d}.png'))

# Data loaders (con augmentation de C3)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomRotation(degrees=5),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 0.5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_dataset = datasets.ImageFolder(os.path.join(BASE_DIR, 'train'), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(BASE_DIR, 'val'),   transform=val_transform)
test_dataset  = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'),  transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Weighted loss
n_normal   = len(os.listdir(os.path.join(BASE_DIR, 'train', 'normal')))
n_arritmia = len(os.listdir(os.path.join(BASE_DIR, 'train', 'arritmia')))
total      = n_normal + n_arritmia
pos_weight = torch.tensor([total / (2 * n_arritmia) / (total / (2 * n_normal))]).to(DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

print('✅ Datos, loaders y criterion listos.')
print(f'   Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')

## PASO 2 — Early Stopping (de C3)

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.001, path='mejor_modelo.pt'):
        self.patience  = patience
        self.min_delta = min_delta
        self.path      = path
        self.counter   = 0
        self.best_loss = None
        self.stop      = False

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            torch.save(model.state_dict(), self.path)
        elif val_loss < self.best_loss - self.min_delta:
            print(f'    ✅ Val loss mejoró: {self.best_loss:.4f} → {val_loss:.4f}')
            self.best_loss = val_loss
            self.counter   = 0
            torch.save(model.state_dict(), self.path)
        else:
            self.counter += 1
            print(f'    ⚠️  Sin mejora ({self.counter}/{self.patience})')
            if self.counter >= self.patience:
                print(f'    🛑 Early stopping activado.')
                self.stop = True
        return self.stop

print('EarlyStopping definido.')

## PASO 3 — Funciones de train y val

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images = images.to(device)
        labels = labels.float().to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).float()
        correct += (preds == labels).sum().item()
        total   += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.float().to(device)
            outputs = model(images).squeeze()
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total   += labels.size(0)
    return total_loss / len(loader), correct / total

print('Funciones de entrenamiento definidas.')

## PASO 4 — FASE 1: Capas congeladas

### ¿Qué significa "congelar" capas?
Congelar una capa = sus pesos NO se actualizan durante el entrenamiento. Solo la nueva capa final aprende. Así la capa nueva se adapta a ECGs sin "desaprender" lo que EfficientNet-B0 sabe de ImageNet.

In [ ]:
# ─── Crear modelo y congelar todo excepto el classifier ──────
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(1280, 1)
model = model.to(DEVICE)

# Congelar TODAS las capas
for param in model.parameters():
    param.requires_grad = False

# Descongelar SOLO el classifier (la capa que reemplazamos)
for param in model.classifier.parameters():
    param.requires_grad = True

# Verificar
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=== FASE 1: Capas congeladas ===')
print(f'Total parámetros:       {total_params:,}')
print(f'Parámetros entrenables: {trainable_params:,}  ← solo el classifier')
print(f'Parámetros congelados:  {total_params - trainable_params:,}')
print()
print('Solo la última capa se actualiza en Fase 1.')

In [ ]:
# ─── Optimizador Fase 1 ───────────────────────────────────────
# lr = 1e-3: relativamente alto, está bien porque solo mueve la capa nueva
# Solo pasamos los parámetros entrenables al optimizador
optimizer_fase1 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)

# ReduceLROnPlateau: reduce lr a la mitad si val_loss no mejora en 3 epochs
scheduler_fase1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fase1,
    mode='min',      # monitorear mínimo (val_loss debe bajar)
    factor=0.5,      # multiplicar lr × 0.5 cuando se activa
    patience=3,      # esperar 3 epochs sin mejora antes de reducir
)

early_stopping_f1 = EarlyStopping(patience=5, path='modelo_fase1.pt')

print('Optimizador, scheduler y early stopping de Fase 1 listos.')
print(f'LR inicial Fase 1: {optimizer_fase1.param_groups[0]["lr"]}')

In [ ]:
# ─── Entrenar Fase 1 ──────────────────────────────────────────
N_EPOCHS_FASE1 = 3  # Con datos reales: 5-10 epochs
history_f1     = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

print('=' * 55)
print('FASE 1 — Solo classifier entrenable')
print('=' * 55)

for epoch in range(1, N_EPOCHS_FASE1 + 1):
    current_lr = optimizer_fase1.param_groups[0]['lr']
    print(f'\n[Epoch {epoch}/{N_EPOCHS_FASE1}] LR: {current_lr:.6f}')

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_fase1, DEVICE)
    val_loss,   val_acc   = val_epoch(model,   val_loader,   criterion, DEVICE)

    history_f1['train_loss'].append(train_loss)
    history_f1['val_loss'].append(val_loss)
    history_f1['train_acc'].append(train_acc)
    history_f1['val_acc'].append(val_acc)
    history_f1['lr'].append(current_lr)

    print(f'  Train → Loss: {train_loss:.4f} | Acc: {train_acc*100:.1f}%')
    print(f'  Val   → Loss: {val_loss:.4f} | Acc: {val_acc*100:.1f}%')

    scheduler_fase1.step(val_loss)   # ReduceLROnPlateau monitorea val_loss

    if early_stopping_f1(val_loss, model):
        break

print()
print('✅ FASE 1 completada.')
print(f'   Modelo guardado en: modelo_fase1.pt')

## PASO 5 — FASE 2: Fine-tuning (descongelar últimas capas)

### ¿Qué cambia en Fase 2?
Descongelamos las **últimas 3 capas** de `model.features` para que también se afinen. Usamos un lr mucho más bajo (1e-5) para no destruir el conocimiento previo — solo hacemos ajustes finos.

In [ ]:
# ─── Cargar el mejor modelo de Fase 1 ────────────────────────
model.load_state_dict(torch.load('modelo_fase1.pt', map_location=DEVICE))
print('Mejor modelo de Fase 1 cargado.')

# ─── Descongelar las últimas 3 capas de features ─────────────
# Primero verificar cuántas capas tiene features
print(f'\nTotal de bloques en model.features: {len(model.features)}')
print('Bloques (índice → nombre):')
for i, (name, _) in enumerate(model.features.named_children()):
    print(f'  [{i}] {name}')

In [ ]:
# Descongelar los últimos 3 bloques de features + el classifier
# Primero recongelar todo
for param in model.parameters():
    param.requires_grad = False

# Descongelar últimos 3 bloques de features
features_list = list(model.features.children())
for bloque in features_list[-3:]:
    for param in bloque.parameters():
        param.requires_grad = True

# Descongelar classifier
for param in model.classifier.parameters():
    param.requires_grad = True

# Verificar
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=== FASE 2: Últimas capas descongeladas ===')
print(f'Total parámetros:       {total_params:,}')
print(f'Parámetros entrenables: {trainable_params:,}  ← classifier + últimos 3 bloques')
print(f'Parámetros congelados:  {total_params - trainable_params:,}')

In [ ]:
# ─── Optimizador Fase 2 ───────────────────────────────────────
# lr = 1e-5: MUY bajo para no destruir los pesos pre-entrenados
optimizer_fase2 = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-5
)

scheduler_fase2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_fase2,
    mode='min',
    factor=0.5,
    patience=3,
)

early_stopping_f2 = EarlyStopping(patience=5, path='modelo_fase2.pt')

print(f'LR inicial Fase 2: {optimizer_fase2.param_groups[0]["lr"]}')
print('(100x más bajo que Fase 1 — ajustes muy finos)')

In [ ]:
# ─── Entrenar Fase 2 ──────────────────────────────────────────
N_EPOCHS_FASE2 = 3  # Con datos reales: 10-20 epochs
history_f2     = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}

print('=' * 55)
print('FASE 2 — Fine-tuning (últimas capas descongeladas)')
print('=' * 55)

for epoch in range(1, N_EPOCHS_FASE2 + 1):
    current_lr = optimizer_fase2.param_groups[0]['lr']
    print(f'\n[Epoch {epoch}/{N_EPOCHS_FASE2}] LR: {current_lr:.8f}')

    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer_fase2, DEVICE)
    val_loss,   val_acc   = val_epoch(model,   val_loader,   criterion, DEVICE)

    history_f2['train_loss'].append(train_loss)
    history_f2['val_loss'].append(val_loss)
    history_f2['train_acc'].append(train_acc)
    history_f2['val_acc'].append(val_acc)
    history_f2['lr'].append(current_lr)

    print(f'  Train → Loss: {train_loss:.4f} | Acc: {train_acc*100:.1f}%')
    print(f'  Val   → Loss: {val_loss:.4f} | Acc: {val_acc*100:.1f}%')

    scheduler_fase2.step(val_loss)

    if early_stopping_f2(val_loss, model):
        break

print()
print('✅ FASE 2 completada.')
print(f'   Mejor modelo final guardado en: modelo_fase2.pt')

## PASO 6 — Curvas de entrenamiento (Fase 1 + Fase 2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Transfer Learning — Fase 1 (congelado) + Fase 2 (fine-tuning)', fontsize=12)

e1 = list(range(1, len(history_f1['train_loss']) + 1))
e2 = list(range(len(e1) + 1, len(e1) + len(history_f2['train_loss']) + 1))

# Loss
axes[0].plot(e1, history_f1['train_loss'], 'b-o', label='Train F1')
axes[0].plot(e1, history_f1['val_loss'],   'b--o', label='Val F1')
axes[0].plot(e2, history_f2['train_loss'], 'r-o', label='Train F2')
axes[0].plot(e2, history_f2['val_loss'],   'r--o', label='Val F2')
axes[0].axvline(x=len(e1) + 0.5, color='gray', linestyle=':', label='Inicio F2')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(e1, [a*100 for a in history_f1['train_acc']], 'b-o', label='Train F1')
axes[1].plot(e1, [a*100 for a in history_f1['val_acc']],   'b--o', label='Val F1')
axes[1].plot(e2, [a*100 for a in history_f2['train_acc']], 'r-o', label='Train F2')
axes[1].plot(e2, [a*100 for a in history_f2['val_acc']],   'r--o', label='Val F2')
axes[1].axvline(x=len(e1) + 0.5, color='gray', linestyle=':', label='Inicio F2')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('Accuracy')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('curvas_transfer_learning.png', dpi=100, bbox_inches='tight')
plt.show()
print('Con datos reales, esperamos ver una mejora notable al pasar de Fase 1 a Fase 2.')

## PASO 7 — Resumen del pipeline completo para Semana 3

In [ ]:
print("""
PIPELINE COMPLETO — LISTO PARA DATOS REALES (SEMANA 3)
=======================================================

Cuando Gloria entregue las imágenes, solo hay que cambiar:

  BASE_DIR = 'ecg_dataset_dummy'
         ↓
  BASE_DIR = '/content/drive/MyDrive/ecg_images'  ← ruta real en Drive

Y ajustar el número de epochs:
  N_EPOCHS_FASE1 = 3  →  N_EPOCHS_FASE1 = 10
  N_EPOCHS_FASE2 = 3  →  N_EPOCHS_FASE2 = 20

Todo lo demás (modelo, augmentation, weighted loss,
early stopping, scheduler) funciona igual.

RESUMEN DE COMPONENTES IMPLEMENTADOS
=====================================
C2 ✅  EfficientNet-B0 adaptado (Linear 1280→1)
C2 ✅  DataLoader con ImageFolder
C2 ✅  Normalización ImageNet
C3 ✅  Data augmentation (rotación, brillo, traslación, blur)
C3 ✅  Weighted loss (BCEWithLogitsLoss + pos_weight)
C3 ✅  Early stopping (patience=5)
C4 ✅  Fase 1: capas congeladas, lr=1e-3
C4 ✅  Fase 2: fine-tuning últimas capas, lr=1e-5
C4 ✅  ReduceLROnPlateau (factor=0.5, patience=3)
""")

## ✅ Checklist C4

In [ ]:
print("""
✅ CHECKLIST C4
================

[✓] Fase 1 implementada:
    → Todas las capas congeladas excepto classifier
    → Adam lr=1e-3
    → 3 epochs de prueba sin errores
    → Mejor modelo guardado en modelo_fase1.pt

[✓] Fase 2 implementada:
    → Últimas 3 capas de features descongeladas
    → Adam lr=1e-5 (fine-tuning suave)
    → 3 epochs de prueba sin errores
    → Mejor modelo guardado en modelo_fase2.pt

[✓] ReduceLROnPlateau activo en ambas fases
[✓] Early stopping activo en ambas fases
[✓] Curvas de entrenamiento graficadas

──────────────────────────────────────────
SEMANA 2 COMPLETADA ✓
──────────────────────────────────────────

Semana 3 (cuando lleguen las imágenes de Gloria):
  → C5: Cambiar BASE_DIR, aumentar epochs, entrenar en serio
  → C6: Evaluar, métricas, matriz de confusión, exportar ONNX
""")